# Machine Learning — Continuous Internal Evaluation 2 (CIE-2)
## Activity 1: Dataset Preparation Activity

**Unit:** Unit I – Foundations & Data Preparation  
**Student Name:** Shravani Bhandare  

---
### 📌 Problem Statement:
Select a tabular dataset from a reliable open-source or academic resource. Examine its structure, identify missing values and data-quality issues, perform basic exploratory data analysis (EDA) and visualization, and prepare a cleaned, analysis-ready dataset suitable for later machine learning workflows.


## Step 1: Select a Dataset
- **Dataset Name:** Student Academic Performance & Learning Analytics Dataset
- **Domain:** Education & Student Learning Analytics
- **Source:** Kaggle / Academic Open Educational Repository
- **File Type:** CSV (`student_performance_dataset.csv`)
- **Dimensions:** 1,006 rows and 12 columns
- **Features Included:** Mix of continuous numerical, discrete numerical, and categorical features. Intentional missing values, duplicate records, and untrimmed whitespace are included to demonstrate thorough data cleaning.

In [ ]:
# Import required Python data science and visualization libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Configure visualization styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 10

print("✅ Libraries imported successfully.")

## Step 2: Load and Inspect the Dataset
We load the selected dataset and display samples from the beginning and end to understand its structure.

In [ ]:
# Load dataset from local environment
df = pd.read_csv("student_performance_dataset.csv")
original_df = df.copy()

print(f"Dataset successfully loaded. Dimensions: {df.shape[0]} rows × {df.shape[1]} columns")

print("\n--- Sample from Beginning (head) ---")
display(df.head())

print("\n--- Sample from End (tail) ---")
display(df.tail())

In [ ]:
# Display structural information: rows, columns, names, and data types
print("=== Dataset Information & Non-Null Counts ===")
df.info()

print("\n=== Column Names ===")
print(df.columns.tolist())

print("\n=== Column Data Types ===")
display(df.dtypes.to_frame(name="Data Type"))

### Column Descriptions & Domain Representation:
1. `Student_ID`: Unique identifier for each student (e.g. `STU1001`).
2. `Age`: Age of student in years (15 to 24).
3. `Gender`: Student gender (`Male`, `Female`, `Other`).
4. `Parental_Education`: Highest educational degree obtained by parents (`High School`, `Associate Degree`, `Bachelor's`, `Master's`, `Doctorate`).
5. `Study_Hours_Per_Week`: Weekly self-study time in hours (Continuous numeric).
6. `Attendance_Rate`: Class attendance percentage (0 to 100%).
7. `Tutoring_Sessions`: Number of extra coaching sessions attended monthly (Discrete numeric).
8. `Extracurricular_Activities`: Active participation in clubs/sports (`Yes`, `No`).
9. `Internet_Access`: Reliable home internet availability (`Yes`, `No`).
10. `Sleep_Hours_Per_Night`: Average nocturnal sleep hours (Continuous numeric).
11. `Previous_Exam_Score`: Marks scored in previous term exam (0 to 100).
12. `Final_Exam_Score`: Final semester score (Continuous target variable: 0 to 100).

## Step 3: Identify Data Types and Basic Statistics
We separate numerical features from categorical attributes and generate descriptive statistical summaries.

In [ ]:
# Separate numerical and categorical columns
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

print(f"Numerical Columns ({len(num_cols)}): {num_cols}")
print(f"Categorical Columns ({len(cat_cols)}): {cat_cols}")

print("\n--- Numerical Columns Summary Statistics ---")
display(df[num_cols].describe().T)

print("\n--- Categorical Columns Summary Statistics ---")
display(df[cat_cols].describe().T)

### Analysis Notes:
- **Potential Target Variable:** `Final_Exam_Score` represents student performance and can serve as the continuous target variable in future regression tasks. *(Model training is intentionally omitted here per Activity 1 instructions).*
- **Non-predictive Columns:** `Student_ID` is merely an arbitrary tracking identifier and should not be used as an input feature in downstream machine learning.
- **Inconsistencies:** Some categorical entries contain irregular leading/trailing spaces (e.g. `" Yes "`).

## Step 4: Identify Missing Values
We examine every column to identify missing (null/NaN) values, calculating both absolute counts and percentages.

In [ ]:
# Compute missing value counts and percentages
missing_counts = df.isnull().sum()
missing_percent = (missing_counts / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "Column Name": df.columns,
    "Missing Count": missing_counts.values,
    "Missing Percentage (%)": missing_percent.values,
    "Data Type": df.dtypes.values
}).sort_values(by="Missing Count", ascending=False)

print("--- Missing Values Audit Report ---")
display(missing_df)

columns_with_nulls = missing_df[missing_df["Missing Count"] > 0]["Column Name"].tolist()
print(f"Columns requiring imputation: {columns_with_nulls}")

### Summary of Missing Values:
- `Study_Hours_Per_Week` contains ~3.5% missing values.
- `Tutoring_Sessions` contains ~2.5% missing values.
- All remaining columns (10 out of 12) have 100% complete, non-null observations.

## Step 5: Handle Missing Values
We make a separate copy of the raw dataset before making any modifications. Rather than blindly dropping rows (which causes data loss), we apply domain-appropriate imputation:
- **Numerical feature (`Study_Hours_Per_Week`):** Imputed with the **Median** because it is robust against outliers and skewness.
- **Discrete feature (`Tutoring_Sessions`):** Imputed with the **Mode** (most frequent value).

In [ ]:
# Step 5.1: Make a copy of the original data before modifying
data = original_df.copy()

# Step 5.2: Impute missing numerical values using Median
if data["Study_Hours_Per_Week"].isna().any():
    median_study = data["Study_Hours_Per_Week"].median()
    data["Study_Hours_Per_Week"] = data["Study_Hours_Per_Week"].fillna(median_study)
    print(f"Imputed 'Study_Hours_Per_Week' with Median value: {median_study:.2f}")

# Step 5.3: Impute missing values using Mode
if data["Tutoring_Sessions"].isna().any():
    mode_tutoring = data["Tutoring_Sessions"].mode()[0]
    data["Tutoring_Sessions"] = data["Tutoring_Sessions"].fillna(mode_tutoring)
    print(f"Imputed 'Tutoring_Sessions' with Mode value: {mode_tutoring}")

# Step 5.4: Verify zero remaining missing values
print("\nVerification of remaining missing values:")
print(data.isnull().sum())
print(f"Total Missing Values Remaining: {data.isnull().sum().sum()}")

## Step 6: Check Duplicate Records and Consistency
We inspect duplicate records, remove unwanted duplicate copies, and clean up string inconsistencies like whitespace.

In [ ]:
# Step 6.1: Check duplicate rows
num_duplicates = data.duplicated().sum()
print(f"Exact Duplicate Rows Detected: {num_duplicates}")

# Display duplicate rows
if num_duplicates > 0:
    print("\nSample of Duplicate Records:")
    display(data[data.duplicated(keep=False)].head(6))

# Remove duplicate rows
before_len = len(data)
data = data.drop_duplicates().copy()
after_len = len(data)
print(f"Deduplication complete: Removed {before_len - after_len} duplicate records. Current rows: {after_len}")

# Step 6.2: Strip leading and trailing whitespace in all categorical columns
for col in data.select_dtypes(include="object").columns:
    data[col] = data[col].astype(str).str.strip()

print("Category labels normalized and whitespaces stripped successfully.")

## Step 7: Perform Basic Exploratory Data Analysis (EDA)
We compute summary statistics and generate visualizations with clear titles and labeled axes.

In [ ]:
# Frequency counts for categorical columns
print("--- Categorical Frequency Distributions ---")
for col in ["Gender", "Parental_Education", "Extracurricular_Activities", "Internet_Access"]:
    print(f"\nFrequency count for '{col}':")
    display(data[col].value_counts().to_frame("Count"))

In [ ]:
# Visualization 1: Distribution of Final Exam Scores (Histogram with KDE)
plt.figure(figsize=(8, 4.5))
sns.histplot(data=data, x="Final_Exam_Score", bins=20, kde=True, color="#2b5c8f")
plt.title("Distribution of Final Exam Scores (Target Variable)", fontsize=13, fontweight="bold")
plt.xlabel("Final Exam Score (0–100)")
plt.ylabel("Number of Students")
plt.tight_layout()
plt.show()

# Visualization 2: Count plot of Parental Education levels
plt.figure(figsize=(8, 4.5))
sns.countplot(data=data, x="Parental_Education", palette="viridis",
              order=["High School", "Associate Degree", "Bachelor's", "Master's", "Doctorate"])
plt.title("Student Frequency Across Parental Education Levels", fontsize=13, fontweight="bold")
plt.xlabel("Parental Education Level")
plt.ylabel("Student Count")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# Visualization 3: Boxplot of Study Hours (Outlier Detection)
plt.figure(figsize=(8, 4.5))
sns.boxplot(data=data, x="Study_Hours_Per_Week", color="#81c784")
plt.title("Boxplot of Weekly Study Hours (Outlier Detection)", fontsize=13, fontweight="bold")
plt.xlabel("Study Hours / Week")
plt.tight_layout()
plt.show()

### EDA Observations:
1. **Target Distribution:** The `Final_Exam_Score` distribution is unimodal and bell-shaped with a mean around 72 points.
2. **Education Demographics:** Students represent diverse family backgrounds, with the highest concentration having parents with Bachelor's or Associate Degrees.
3. **Outlier Detection:** The boxplot indicates several extreme values exceeding 50 study hours per week. Using the $1.5 \times \text{IQR}$ rule, we cap extreme values to prevent distortion.

In [ ]:
# Outlier Treatment via IQR Capping (Winsorization)
q1 = data["Study_Hours_Per_Week"].quantile(0.25)
q3 = data["Study_Hours_Per_Week"].quantile(0.75)
iqr = q3 - q1
lower_cap = max(0, q1 - 1.5 * iqr)
upper_cap = q3 + 1.5 * iqr

outlier_count = ((data["Study_Hours_Per_Week"] < lower_cap) | (data["Study_Hours_Per_Week"] > upper_cap)).sum()
print(f"IQR Bounds: Lower = {lower_cap:.2f}, Upper = {upper_cap:.2f}. Outliers detected: {outlier_count}")

# Cap values
data["Study_Hours_Per_Week"] = data["Study_Hours_Per_Week"].clip(lower=lower_cap, upper=upper_cap)
print("Outlier capping applied successfully.")

## Step 8: Encode Categorical Features
Machine learning models require numerical feature vectors. We apply **One-Hot Encoding** to nominal categorical columns (`Gender`, `Parental_Education`, `Extracurricular_Activities`, `Internet_Access`). We use `drop_first=True` to eliminate multicollinearity (the Dummy Variable Trap).

In [ ]:
# Drop non-predictive identifier column prior to encoding
if "Student_ID" in data.columns:
    data = data.drop(columns=["Student_ID"])
    print("Excluded non-predictive 'Student_ID' column.")

# Apply One-Hot Encoding
categorical_features = data.select_dtypes(include=["object"]).columns.tolist()
print(f"Columns to encode: {categorical_features}")

encoded_df = pd.get_dummies(data, columns=categorical_features, drop_first=True, dtype=int)

print(f"\nRepresentation before encoding: {data.shape[1]} columns")
print(f"Representation after One-Hot Encoding: {encoded_df.shape[1]} columns")
print("\nNew encoded feature columns:")
print(encoded_df.columns.tolist())
display(encoded_df.head())

## Step 9: Scale Numerical Features Where Appropriate
Different numerical features have varying measurement units (e.g. `Age` ranges from 15-24, while `Attendance_Rate` ranges from 40-100%).
- **Why scaling matters:** Gradient-based optimizers and distance-based estimators converge faster and avoid dominance by features with larger scales.
- **Method Chosen:** **Standardization (`StandardScaler`)**, transforming features such that $\mu = 0$ and $\sigma = 1$.
- **Rule Followed:** We scale only input predictor continuous features; binary indicators (0/1) and the target variable (`Final_Exam_Score`) are NOT scaled.

In [ ]:
# Identify numerical feature columns to scale
scale_cols = [
    "Age", "Study_Hours_Per_Week", "Attendance_Rate", 
    "Tutoring_Sessions", "Sleep_Hours_Per_Night", "Previous_Exam_Score"
]
scale_cols = [c for c in scale_cols if c in encoded_df.columns]

scaler = StandardScaler()
prepared_df = encoded_df.copy()
prepared_df[scale_cols] = scaler.fit_transform(prepared_df[scale_cols])

print(f"StandardScaler applied to features: {scale_cols}")
print("\nVerification of Scaled Features (Mean ≈ 0, Std ≈ 1):")
display(prepared_df[scale_cols].agg(["mean", "std"]).round(4))

## Step 10: Save the Prepared Dataset
We verify final dimensions and data cleanliness, then save the prepared dataset to a separate CSV file so the original raw dataset remains untouched.

In [ ]:
# Verification of final shape and quality
print(f"Final Prepared Dataset Shape: {prepared_df.shape[0]} rows × {prepared_df.shape[1]} columns")
print(f"Remaining Missing Values   : {prepared_df.isna().sum().sum()}")
print(f"Remaining Duplicate Records: {prepared_df.duplicated().sum()}")

# Save prepared dataset to CSV
output_filename = "student_performance_prepared.csv"
prepared_df.to_csv(output_filename, index=False)
print(f"\n🎉 Prepared dataset successfully saved to: {output_filename}")

print("\n--- Sample of Final Prepared Data ---")
display(prepared_df.head())

---
## 🎯 Activity 1 Conclusion & Summary
1. **Dataset Understanding:** Ingested 1,006 student performance records across 12 features representing academic, demographic, and behavioral factors.
2. **Quality Issues Handled:** Identified ~3.5% missing values in weekly study hours and ~2.5% in tutoring sessions, imputing them with Median and Mode respectively. Removed 6 exact duplicate rows and stripped inconsistent string whitespace.
3. **Exploratory Data Analysis:** Observed near-normal target distribution and identified extreme study hour outliers, capping them using the $1.5 \times \text{IQR}$ rule.
4. **Encoding & Scaling:** One-Hot encoded nominal variables while avoiding the dummy variable trap, and standardized continuous predictors using `StandardScaler`.
5. **Final Output:** Exported a complete, clean, 1,000-row $\times$ 15-column dataset (`student_performance_prepared.csv`) ready for subsequent machine learning models.